<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/negatome_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
import random
import json
from datetime import datetime
import os


In [ ]:


# 1. Define file paths
txt_file_path = "/content/combined_stringent.txt"
parquet_file_path = "/content/combined_stringent.parquet"

# 2. Read the text file into a DataFrame (it's a tab-separated file)
df = pd.read_csv(txt_file_path, sep='\t', header=None, names=['protein1', 'protein2'])

# The original code had logic to split a single column, but if it's a tab-separated file,
# reading it directly with `sep='\t'` is more appropriate.
# If the file truly had a single column to be split, the previous logic would be needed,
# but based on the error and the file name, `read_csv` is the correct approach.
# Removing the if df.shape[1] == 1: block as it's no longer necessary with direct CSV read.

# 3. Convert and save to Parquet format with a new name
df.to_parquet(parquet_file_path, engine="pyarrow", compression="snappy")

print(f"Successfully converted and saved data from {txt_file_path} to {parquet_file_path}")

Successfully converted and saved data from /content/combined_stringent.txt to /content/combined_stringent.parquet


##building protein universe

In [ ]:
# Load your existing data
positives = pd.read_parquet('/content/positives_string.parquet')
negatome  = pd.read_parquet('/content/combined_stringent.parquet')

# At this point STRING still uses Ensembl IDs
# We work in Ensembl ID space for now
# (UniProt mapping comes in Step 5)

# Get all unique proteins that appear in your positive pairs
proteins_A = set(positives['protein1'].unique())
proteins_B = set(positives['protein2'].unique())
protein_universe = proteins_A | proteins_B   # union

print(f"Unique proteins in positive set: {len(protein_universe):,}")
# Expect: ~10,000–15,000 unique proteins

Unique proteins in positive set: 3,653


In [ ]:
n_positives = len(positives)
n_negatome  = len(negatome)
target_ratio = 3    # 1 positive : 3 negatives total

n_total_negatives_needed = n_positives * target_ratio
n_random_needed = n_total_negatives_needed - n_negatome

print(f"Positives:               {n_positives:,}")
print(f"Negatome negatives:      {n_negatome:,}")
print(f"Total negatives needed:  {n_total_negatives_needed:,}")
print(f"Random negatives needed: {n_random_needed:,}")
print(f"Protein universe size:   {len(protein_universe):,}")

Positives:               10,405
Negatome negatives:      6,136
Total negatives needed:  31,215
Random negatives needed: 25,079
Protein universe size:   3,653


##exclusion set

In [ ]:
# Load the FULL STRING file — not just your filtered positives
# You need ALL pairs regardless of score to avoid false negatives
string_full = pd.read_csv(
    '/content/9606.protein.links.detailed.v12.0.onlyAB.tsv',
    sep='\t'
)

print(f"Full STRING pairs loaded: {len(string_full):,}")
print(string_full.head(3))

Full STRING pairs loaded: 231,811
               protein1              protein2  neighborhood  fusion  \
0  9606.ENSP00000000233  9606.ENSP00000356607           0.0     0.0   
1  9606.ENSP00000000233  9606.ENSP00000427567           0.0     0.0   
2  9606.ENSP00000000233  9606.ENSP00000253413           0.0     0.0   

   cooccurence  coexpression  experimental  database  textmining  \
0          0.0          45.0         134.0       0.0        81.0   
1          0.0           0.0         128.0       0.0        70.0   
2          0.0         118.0          49.0       0.0        69.0   

   combined_score  
0           173.0  
1           154.0  
2           151.0  


In [ ]:
print("Building exclusion set from full STRING file...")

exclusion_set = set()

for _, row in string_full.iterrows():
    p1 = row['protein1']
    p2 = row['protein2']
    # Store canonical order (smaller string first)
    # so (A,B) and (B,A) are treated as the same pair
    pair = (min(p1, p2), max(p1, p2))
    exclusion_set.add(pair)

print(f"Pairs in exclusion set: {len(exclusion_set):,}")
# Should match full STRING pairs: ~231,811

Building exclusion set from full STRING file...
Pairs in exclusion set: 231,811


In [ ]:
for _, row in positives.iterrows():
    p1 = row['protein1']
    p2 = row['protein2']
    pair = (min(p1, p2), max(p1, p2))
    exclusion_set.add(pair)

print(f"Exclusion set after adding positives: {len(exclusion_set):,}")

Exclusion set after adding positives: 240,414


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)

protein_list     = list(protein_universe)
n_proteins       = len(protein_list)
random_negatives = []
sampled_set      = set()

# Keep going until we have exactly what we need — no attempt cap
while len(random_negatives) < n_random_needed:
    # numpy randint is faster than random.sample for large loops
    i, j = np.random.choice(n_proteins, size=2, replace=False)
    p1, p2 = protein_list[i], protein_list[j]
    pair   = (min(p1, p2), max(p1, p2))

    if pair in exclusion_set:
        continue
    if pair in sampled_set:
        continue

    random_negatives.append({
        'protein1': pair[0],
        'protein2': pair[1],
        'label':    0,
        'source':   'random_negative'
    })
    sampled_set.add(pair)

random_neg_df = pd.DataFrame(random_negatives)
print(f"Random negatives sampled: {len(random_neg_df):,}")
print(random_neg_df.head(3))

Random negatives sampled: 25,079
               protein1              protein2  label           source
0  9606.ENSP00000356669  9606.ENSP00000379350      0  random_negative
1  9606.ENSP00000333283  9606.ENSP00000434359      0  random_negative
2  9606.ENSP00000248572  9606.ENSP00000469613      0  random_negative


In [ ]:
print(f"n_positives:  {n_positives}")
print(f"n_negatome:   {n_negatome}")
print(f"calculation:  {n_positives * 3} - {n_negatome} = {n_positives * 3 - n_negatome}")
print(f"n_random_needed: {n_random_needed}")

n_positives:  10405
n_negatome:   6136
calculation:  31215 - 6136 = 25079
n_random_needed: 25079


In [ ]:
print(f"Negatome pairs:          {n_negatome}")
print(f"Random negatives:        {len(random_neg_df)}")
print(f"Total negatives:         {n_negatome + len(random_neg_df)}")
print(f"Positives:               {n_positives}")
print(f"Exact ratio achieved:    {(n_negatome + len(random_neg_df)) / n_positives:.4f}")
# Should print exactly 3.0000

Negatome pairs:          6136
Random negatives:        25079
Total negatives:         31215
Positives:               10405
Exact ratio achieved:    3.0000


In [ ]:
# Check 1 — no self pairs (a protein paired with itself)
self_pairs = (random_neg_df['protein1'] == random_neg_df['protein2']).sum()
print(f"Self pairs: {self_pairs}")   # must be 0

# Check 2 — no overlap with positives
positive_pairs = set(zip(positives['protein1'], positives['protein2']))
overlap = random_neg_df.apply(
    lambda r: (r['protein1'], r['protein2']) in positive_pairs
              or (r['protein2'], r['protein1']) in positive_pairs,
    axis=1
).sum()
print(f"Overlap with positives: {overlap}")   # must be 0

# Check 3 — no duplicate pairs within random negatives
duplicates = random_neg_df.duplicated(subset=['protein1', 'protein2']).sum()
print(f"Duplicate pairs: {duplicates}")   # must be 0

# Check 4 — all proteins come from the universe
all_proteins_in_sample = set(random_neg_df['protein1']) | set(random_neg_df['protein2'])
outside_universe = all_proteins_in_sample - protein_universe
print(f"Proteins outside universe: {len(outside_universe)}")   # must be 0

Self pairs: 0
Overlap with positives: 0
Duplicate pairs: 0
Proteins outside universe: 0


In [ ]:
import json
from datetime import datetime
import os

# Save
random_neg_df.to_csv('/content/negatives_random.tsv', sep='\t', index=False)
random_neg_df.to_parquet('/content/negatives_random.parquet', index=False)

# Audit trail
metadata = {
    "date_generated":        datetime.today().strftime('%Y-%m-%d'),
    "random_seed":           42,
    "protein_universe_size": len(protein_universe),
    "n_positives":           n_positives,
    "n_negatome":            n_negatome,
    "target_ratio":          3,
    "n_random_sampled":      len(random_neg_df),
    "total_negatives":       n_negatome + len(random_neg_df),
    "ratio_achieved":        round((n_negatome + len(random_neg_df)) / n_positives, 4),
    "exclusion_source":      "STRING physical links all scores — 231,811 pairs",
    "label":                 0,
    "checks_passed": {
        "self_pairs":               0,
        "overlap_with_positives":   0,
        "duplicate_pairs":          0,
        "proteins_outside_universe":0
    }
}

# Create the directory if it doesn't exist
os.makedirs('data/raw', exist_ok=True)

with open('data/raw/negatives_random_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_generated": "2026-06-16",
  "random_seed": 42,
  "protein_universe_size": 3653,
  "n_positives": 10405,
  "n_negatome": 6136,
  "target_ratio": 3,
  "n_random_sampled": 25079,
  "total_negatives": 31215,
  "ratio_achieved": 3.0,
  "exclusion_source": "STRING physical links all scores \u2014 231,811 pairs",
  "label": 0,
  "checks_passed": {
    "self_pairs": 0,
    "overlap_with_positives": 0,
    "duplicate_pairs": 0,
    "proteins_outside_universe": 0
  }
}
